In [1]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from torch.utils.data import Dataset, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# CONFIG
d_model = 256
num_heads = 8
num_layers = 5
d_ff = 1024
vocab_size = 30000

In [4]:
import torch
import torch.nn as nn
class TransformerClassifier(nn.Module):
    def __init__(self, base_model, d_model, num_classes=4):
        super().__init__()
        self.embedding = base_model.embedding
        self.pe_module = base_model.pe_module
        self.layers = base_model.layers
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        seq_len = x.size(1)
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0).to(x.device)

        x = self.embedding(x)
        x, rotary_freqs = self.pe_module(x)

        for layer in self.layers:
            attn_out, _ = layer['mha'](x, x, x, mask, rotary_freqs)
            x = layer['norm1'](x + layer['dropout'](attn_out))

            ffn_out = layer['ffn'](x)
            x = layer['norm2'](x + layer['dropout'](ffn_out))

        pooled = x.mean(dim=1)
        return self.fc(pooled)

In [5]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) 

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :], None 

class LearnedPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0).expand(x.size(0), -1)
        return x + self.pe(positions), None

class RotaryPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x):
        t = torch.arange(x.size(1), device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        return x, torch.cat((freqs, freqs), dim=-1)

def apply_rotary_pos_emb(q, k, freqs):
    cos = freqs.cos().unsqueeze(0).unsqueeze(0)
    sin = freqs.sin().unsqueeze(0).unsqueeze(0)
    def rotate_half(x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat((-x2, x1), dim=-1)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None, rotary_freqs=None):
        batch_size = q.size(0)
        
        # Memory contiguity fixes are applied here
        q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2).contiguous()
        k = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2).contiguous()
        v = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2).contiguous()
        if rotary_freqs is not None:
            q, k = apply_rotary_pos_emb(q, k, rotary_freqs[..., :self.d_k])
            q, k = q.contiguous(), k.contiguous()

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.w_o(out), attn

In [6]:
class SwappablePETransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, pe_module):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pe_module = pe_module
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'mha': MultiHeadAttention(d_model, num_heads),
                'norm1': nn.LayerNorm(d_model),
                'ffn': nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)),
                'norm2': nn.LayerNorm(d_model),
                'dropout': nn.Dropout(0.1)
            }) for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        # Causal mask for language modeling
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0).to(x.device)
        
        x = self.embedding(x)
        x, rotary_freqs = self.pe_module(x)
        
        for layer in self.layers:
            attn_out, _ = layer['mha'](x, x, x, mask, rotary_freqs)
            x = layer['norm1'](x + layer['dropout'](attn_out))
            ffn_out = layer['ffn'](x)
            x = layer['norm2'](x + layer['dropout'](ffn_out))
            
        return self.lm_head(x)

In [7]:
model = SwappablePETransformer(
    vocab_size = 30000,
    d_model = 256,
    num_heads = 8,
    num_layers = 5,
    d_ff = 1024,
    pe_module = RotaryPE(256)
)  # define fully
model.load_state_dict(torch.load('/kaggle/input/models/dhruxp/transformerhp1/pytorch/default/1/custom_model_weights.pth'))
model.to(device)

SwappablePETransformer(
  (embedding): Embedding(30000, 256)
  (pe_module): RotaryPE()
  (layers): ModuleList(
    (0-4): 5 x ModuleDict(
      (mha): MultiHeadAttention(
        (w_q): Linear(in_features=256, out_features=256, bias=True)
        (w_k): Linear(in_features=256, out_features=256, bias=True)
        (w_v): Linear(in_features=256, out_features=256, bias=True)
        (w_o): Linear(in_features=256, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (lm_head): Linear(in_features=256, out_features=30000, bias=True)
)

In [8]:
model.load_state_dict(torch.load('/kaggle/input/models/dhruxp/transformerhp1/pytorch/default/1/custom_model_weights.pth'))
model.to(device)

SwappablePETransformer(
  (embedding): Embedding(30000, 256)
  (pe_module): RotaryPE()
  (layers): ModuleList(
    (0-4): 5 x ModuleDict(
      (mha): MultiHeadAttention(
        (w_q): Linear(in_features=256, out_features=256, bias=True)
        (w_k): Linear(in_features=256, out_features=256, bias=True)
        (w_v): Linear(in_features=256, out_features=256, bias=True)
        (w_o): Linear(in_features=256, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (lm_head): Linear(in_features=256, out_features=30000, bias=True)
)

In [9]:
print(next(model.parameters()).device)

cuda:0


In [10]:
import pandas as pd
train_df = pd.read_csv('/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/train.csv')
test_df  = pd.read_csv('/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/test.csv')
train_df = train_df.fillna("")
test_df  = test_df.fillna("")

In [11]:
train_texts = (train_df['Title'] + " " + train_df['Description']).tolist()
test_texts  = (test_df['Title'] + " " + test_df['Description']).tolist()
train_labels = train_df['Class Index'].tolist()
test_labels  = test_df['Class Index'].tolist()

In [12]:
def tokenizer(text):
    return text.lower().split()
from collections import Counter
counter = Counter()
for text in train_texts:
    counter.update(tokenizer(text))
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= 2:
        vocab[word] = len(vocab)

In [13]:
def numericalize(text):
    return [
        max(0, min(vocab.get(tok, 1), 29999))
        for tok in tokenizer(text)
    ]

In [14]:
import torch
MAX_LEN = 128
def numericalize(text):
    return [vocab.get(tok, vocab["<unk>"]) for tok in tokenizer(text)]
def encode(texts, labels):
    X, y = [], []
    for text, label in zip(texts, labels):
        tokens = torch.tensor(numericalize(text), dtype=torch.long)[:MAX_LEN]
        X.append(tokens)
        y.append(label - 1)
    return X, torch.tensor(y)
train_X, train_y = encode(train_texts, train_labels)
test_X, test_y   = encode(test_texts, test_labels)

In [15]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
class AGNewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True)
    labels = torch.tensor(labels)
    return texts.to(device), labels.to(device)
train_dataset = AGNewsDataset(train_X, train_y)
test_dataset  = AGNewsDataset(test_X, test_y)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

In [16]:
classifier = TransformerClassifier(model, d_model).to(device)

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=3e-4)
def train_epoch():
    classifier.train()
    total_loss = 0
    for src, labels in train_loader:
        src = torch.clamp(src, min=0, max=29999)
        optimizer.zero_grad()
        logits = classifier(src)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

In [18]:
def evaluate():
    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for src, labels in test_loader:
            src = torch.clamp(src, min=0, max=29999)
            logits = classifier(src)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [19]:
for epoch in range(5):
    loss = train_epoch()
    acc = evaluate()
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Acc={acc:.4f}")

Epoch 1: Loss=0.3904, Acc=0.8932
Epoch 2: Loss=0.2521, Acc=0.8995
Epoch 3: Loss=0.2023, Acc=0.9068
Epoch 4: Loss=0.1673, Acc=0.9054
Epoch 5: Loss=0.1414, Acc=0.9021
